# 

In [1]:
import os
import json
import numpy as np

from scipy.optimize import linear_sum_assignment
from pymongo import MongoClient

m2 = 0
a2 = 0

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea


def best_iou_matching(boxesA, boxesB, threshold=0.4):
    if not boxesA or not boxesB:
        return 0

    nA, nB = len(boxesA), len(boxesB)
    iou_matrix = np.zeros((nA, nB))

    for i in range(nA):
        for j in range(nB):
            iou_matrix[i, j] = iou(boxesA[i], boxesB[j])

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matches = sum(iou_matrix[i, j] >= threshold for i, j in zip(row_ind, col_ind))
    return matches


def compute_inter_annotator_agreement(annotation_dir, iou_threshold=0.4):

    annotators = []
    annotator_names = []
    for file in os.listdir(annotation_dir):
        if file.endswith(".json"):
            with open(os.path.join(annotation_dir, file), "r") as f:
                data = json.load(f)
                annotators.append(data["annotations"])
                annotator_names.append(file)
                
    if len(annotators) < 2:
        raise ValueError("Need at least two annotators to compute agreement.")

    video_folders = sorted(set(a["videoFolder"] for ann in annotators for a in ann))
    print(video_folders)
    per_sample = []

    for folder in video_folders:
        sample_boxes = []
        for ann in annotators:
            boxes = []
            for entry in ann:
                if entry["videoFolder"] == folder:
                    boxes = [g["bbox"] for g in entry["groups"] if g["confidence"] >= 1]
                    break
            sample_boxes.append(boxes)

        #print(sample_boxes)

        global m2
        global a2
        pair_agreements = []
        for i in range(len(sample_boxes)):
            for j in range(i + 1, len(sample_boxes)):
                boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                #print(boxesA)
                #print(boxesB)
                matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                m2+=matches
                avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                pair_agreements.append(matches / avg_boxes)
                a2+=avg_boxes
        
        #print(pair_agreements)
        agreement = np.mean(pair_agreements) if pair_agreements else 1.0
        if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
            agreement = 1
        per_sample.append((folder, agreement))

    avg_agreement = np.mean([a for _, a in per_sample]) if per_sample else 1.0

    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")
    for folder, score in per_sample:
        print(f"  {folder:25s}  {score:.3f}")

    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
    print(f"Compared {len(annotators)} annotators: {annotator_names}")

    print(m2/a2)
    
    return per_sample, avg_agreement


if __name__ == "__main__":
    
    MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
    MONGO_DB = 'video_annotations'
    
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]

    annotations_collection = db['annotations']
    yes_no_annotations_collection = db['yes_no_annotations']

    
    pipeline_average_time = [{"$group": {"_id": None,"avgAnnotationDuration": { "$avg": "$annotationDuration"}, "maxAnnotationDuration": { "$max": "$annotationDuration" }, "minAnnotationDuration": { "$min": "$annotationDuration" }, "medianAnnotationDuration": {"$percentile": {"input": "$annotationDuration","p": [0.5],"method": "approximate"}}}}]    
    general_statistics = list(annotations_collection.aggregate(pipeline_average_time))
    
    print('Number of total annotations:', annotations_collection.count_documents({}))
    print('')
    
    #print('Number of total annotations:', annotations_collection.count_documents({}))

    print('')
    print('Average Annotation Time (s): ', general_statistics[0]['avgAnnotationDuration']/1000)
    print('Maximum Annotation Time (s): ', general_statistics[0]['maxAnnotationDuration']/1000)
    print('Minimum Annotation Time (s): ', general_statistics[0]['minAnnotationDuration']/1000)
    print('Median Annotation Time (s): ', general_statistics[0]['medianAnnotationDuration'][0]/1000)
    print('')
    #print(A)

    pipeline_stats_per_annotator = [
        {
            "$group": {
                "_id": "$annotator_id",
                "avgMs": { "$avg": "$annotationDuration" },
                "minMs": { "$min": "$annotationDuration" },
                "maxMs": { "$max": "$annotationDuration" },
                "medianMs": {
                    "$percentile": {
                        "input": "$annotationDuration",
                        "p": [0.5],
                        "method": "approximate"
                    }
                },
                "count": { "$sum": 1 }
            }
        },
        {
            "$project": {
                "avgAnnDuration": {
                    "$round": [
                        { "$divide": ["$avgMs", 1000] },
                        2
                    ]
                },
                "minAnnDuration": {
                    "$round": [
                        { "$divide": ["$minMs", 1000] },
                        2
                    ]
                },
                "maxAnnDuration": {
                    "$round": [
                        { "$divide": ["$maxMs", 1000] },
                        2
                    ]
                },
                "medianAnnotationDuration": {
                    "$round": [
                        {
                            "$divide": [
                                { "$arrayElemAt": ["$medianMs", 0] },
                                1000
                            ]
                        },
                        2
                    ]
                },
                "count": 1
            }
        },
        {
            "$sort": {
                "count": -1
            }
        }
    ]


    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_per_annotator)
    )

    print('Number of distinct annotators:', len(stats_per_annotator))
    
    for stat_per_annotator in stats_per_annotator:
        print(stat_per_annotator)

    pipeline_stats_by_globalIndex_mod3 = [
      {
        "$addFields": {
          "globalIndexMod3": { "$mod": ["$globalIndex", 3] }
        }
      },
      {
        "$group": {
          "_id": "$globalIndexMod3",
          "avgMs": { "$avg": "$annotationDuration" },
          "minMs": { "$min": "$annotationDuration" },
          "maxMs": { "$max": "$annotationDuration" },
          "medianMs": {
            "$percentile": {
              "input": "$annotationDuration",
              "p": [0.5],
              "method": "approximate"
            }
          },
          "count": { "$sum": 1 }
        }
      },
      {
        "$project": {
          "_id": 0,
          "globalIndexMod3": "$_id",
          "avgAnnDuration": {
            "$round": [{ "$divide": ["$avgMs", 1000] }, 2]
          },
          "minAnnDuration": {
            "$round": [{ "$divide": ["$minMs", 1000] }, 2]
          },
          "maxAnnDuration": {
            "$round": [{ "$divide": ["$maxMs", 1000] }, 2]
          },
          "medianAnnDuration": {
            "$round": [
              {
                "$divide": [
                  { "$arrayElemAt": ["$medianMs", 0] },
                  1000
                ]
              },
              2
            ]
          },
          "count": 1
        }
      },
      {
        "$sort": { "globalIndexMod3": 1 }
      }
    ]

    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_by_globalIndex_mod3)
    )

    print('')
    print('Scattered Videos:', stats_per_annotator[1])
    print('Semi-crowded Videos:', stats_per_annotator[2])
    print('Crowded Videos:',stats_per_annotator[0])
    print('')

    #annotation_dir = "annotations"
    #compute_inter_annotator_agreement(annotation_dir)

    videoFolder = annotations_collection.aggregate([
      { "$group": { "_id": "$videoFolder" } },
      { "$sort": { "_id": 1 } }
    ])

    video_folders = sorted(set([vid['_id'] for vid in list(videoFolder)]))
    per_sample = []
    iou_threshold = 0.5
    from collections import defaultdict


    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")

    
    for folder in video_folders:
        #print(folder)
        test = annotations_collection.aggregate([
            {
                "$match": {
                    "videoFolder": folder,
                }
            },
            { "$unwind": "$groups" },
            {
                "$project": {
                    "_id": 0,
                    "videoFolder": 1,
                    "annotator_id": 1,
                    "annotationFrame": "$videoInfo.annotationFrame",
                    "bbox": "$groups.bbox"
                }
            }
        ])
        
        data = list(test)

        frames = defaultdict(lambda: defaultdict(list))
        
        for item in data:
            frame = item["annotationFrame"]
            annotator = item["annotator_id"]
            frames[frame][annotator].append(item["bbox"])
        
        # Final output: list[ list[ list[bbox] ] ]
        results = [
            list(annotator_groups.values())
            for frame, annotator_groups in sorted(frames.items())
        ]


        choices_frame = list(set([dat['annotationFrame'] for dat in data]))
        choices_frame.sort()

        
        for ind, result in enumerate(results):
            annotators = len(result)
            sample_boxes = []
            if annotators >= 2:
                sample_boxes = []
                for boxes in result:
                    sample_boxes.append(boxes)

    
            global m2
            global a2
            pair_agreements = []
            for i in range(len(sample_boxes)):
                for j in range(i + 1, len(sample_boxes)):
                    boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                    #print(boxesA)
                    #print(boxesB)
                    matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                    m2+=matches
                    avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                    pair_agreements.append(matches / avg_boxes)
                    a2+=avg_boxes
            
            #print(pair_agreements)
            agreement = np.mean(pair_agreements) if pair_agreements else 1.0
            if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
                agreement = 1
            print(f"  {folder:25s} {choices_frame[ind]:5d} {agreement:25.3f}")
            per_sample.append((folder, ind, agreement))
    
        avg_agreement = np.mean([a for _, _, a in per_sample]) if per_sample else 1.0

        
    choiced_frames = [1,21,41]
    #for folder, annFrame, score in per_sample:
    #    print(f"  {folder:25s} {choiced_frames[annFrame]:5d} {score:25.3f}")
    
    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
        #print(f"Compared {len(annotators)} annotators: {annotator_names}")
    
        #print(m2/a2)
        

Number of total annotations: 4793


Average Annotation Time (s):  100.89422845816816
Maximum Annotation Time (s):  15070.929
Minimum Annotation Time (s):  0.732
Median Annotation Time (s):  31.4373125

Number of distinct annotators: 16
{'_id': '45efce7c-be08-4706-9de0-de6ee82a7520', 'count': 1460, 'avgAnnDuration': 75.36, 'minAnnDuration': 1.38, 'maxAnnDuration': 13000.22, 'medianAnnotationDuration': 29.93}
{'_id': '1359b1ec-c92f-4f51-9352-0ce981f20bf3', 'count': 1304, 'avgAnnDuration': 200.56, 'minAnnDuration': 2.54, 'maxAnnDuration': 5146.03, 'medianAnnotationDuration': 121.77}
{'_id': '93037c9e-9ad4-4e9f-92b5-9b7f1c13ea44', 'count': 541, 'avgAnnDuration': 20.08, 'minAnnDuration': 0.74, 'maxAnnDuration': 566.73, 'medianAnnotationDuration': 11.7}
{'_id': '3ba39ca4-91cd-465a-b181-1e36ee28728a', 'count': 528, 'avgAnnDuration': 24.82, 'minAnnDuration': 1.11, 'maxAnnDuration': 681.54, 'medianAnnotationDuration': 16.23}
{'_id': 'a365f28f-0648-44f7-a319-521ac35654c7', 'count': 390, 'avgAnnD

In [2]:
!pip install scipy